In [1]:
using LowLevelFEM
using SparseArrays
using LinearAlgebra

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (cache misses: include_dependency fsize change (2), wrong dep version loaded (4), incompatible header (12))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [2]:
function SystemMatrix_(blocks::Matrix{SystemMatrix})

    nrows, ncols = size(blocks)

    # ----------------------------------------------------------
    # 1) Collect trial problems (column-based order)
    # ----------------------------------------------------------
    trial_problems = Problem[]

    function push_unique!(vec, P)
        P === nothing && return
        if all(q -> q !== P, vec)
            push!(vec, P)
        end
    end

    for j in 1:ncols
        for i in 1:nrows
            blk = blocks[i, j]
            if blk.model !== nothing
                push_unique!(trial_problems, blk.model)
                break
            end
        end
    end

    isempty(trial_problems) &&
        error("No trial Problems found in block matrix.")

    # ----------------------------------------------------------
    # 2) Collect test problems (row-based order)
    # ----------------------------------------------------------
    test_problems = Problem[]

    for i in 1:nrows
        for j in 1:ncols
            blk = blocks[i, j]
            if blk.test_model !== nothing
                push_unique!(test_problems, blk.test_model)
                break
            end
        end
    end

    isempty(test_problems) &&
        error("No test Problems found in block matrix.")

    # ----------------------------------------------------------
    # 3) Field-level square check
    # ----------------------------------------------------------
    if length(trial_problems) != length(test_problems) ||
       any(trial_problems[i] !== test_problems[i]
           for i in eachindex(trial_problems))
        error("Block system is not square in field sense. Trial and test spaces differ.")
    end

    problems = trial_problems

    # ----------------------------------------------------------
    # 4) Compute global offsets (based on trial ordering)
    # ----------------------------------------------------------
    offsets = Vector{Int}(undef, length(problems))
    offsets[1] = 0
    for i in 2:length(problems)
        offsets[i] = offsets[i-1] + ndofs(problems[i-1])
    end

    total_dofs = offsets[end] + ndofs(problems[end])

    # helper
    function problem_index(P)
        for (k, q) in enumerate(problems)
            if q === P
                return k
            end
        end
        error("Problem not found in metadata.")
    end

    # ----------------------------------------------------------
    # 5) Assemble global sparse matrix
    # ----------------------------------------------------------
    I = Int[]
    J = Int[]
    V = Float64[]

    for bi in 1:nrows
        for bj in 1:ncols

            blk = blocks[bi, bj]
            blk.A === nothing && continue
            isempty(blk.A) && continue

            rowP = blk.test_model
            colP = blk.model

            rowP === nothing && error("Block missing test_model.")
            colP === nothing && error("Block missing model.")

            iP = problem_index(rowP)
            jP = problem_index(colP)

            row_offset = offsets[iP]
            col_offset = offsets[jP]

            rows, cols, vals = findnz(blk.A)

            # size consistency check
            if maximum(rows) > ndofs(rowP) ||
               maximum(cols) > ndofs(colP)
                error("Block size mismatch in block ($bi,$bj).")
            end

            for k in eachindex(vals)
                push!(I, row_offset + rows[k])
                push!(J, col_offset + cols[k])
                push!(V, vals[k])
            end
        end
    end

    A_big = sparse(I, J, V, total_dofs, total_dofs)

    return SystemMatrix(A_big, nothing, nothing, problems, offsets)
end

SystemMatrix_ (generic function with 1 method)

In [3]:
"""
    SystemVector

Unified global RHS vector for single-field and multi-field systems.

# Fields
- `a::Vector{Float64}`
    Global RHS vector.

Single-field metadata:
- `model::Union{Problem,Nothing}`
    Problem for single-field case.

Multi-field metadata:
- `problems::Union{Vector{Problem},Nothing}`
    Ordered list of Problems (trial order).
- `offsets::Union{Vector{Int},Nothing}`
    Starting global DOF indices of each Problem.

# Semantics
- If `problems === nothing` → single-field case.
- If `problems !== nothing` → multi-field case.
"""
struct SystemVector_
    a::Vector{Float64}

    # ---- Single-field metadata ----
    model::Union{Problem,Nothing}

    # ---- Block metadata ----
    problems::Union{Vector{Problem},Nothing}
    offsets::Union{Vector{Int},Nothing}
end


# ============================================================
# Single-field constructor
# ============================================================

function SystemVector_(field::Union{ScalarField,VectorField,TensorField})

    # assume single load step
    a = vec(field.a[:, 1])

    return SystemVector(a, field.model, nothing, nothing)
end


# ============================================================
# Multi-field constructor (explicit order)
# ============================================================

"""
    SystemVector(fields::Vector)

Construct global RHS vector from a vector of
ScalarField / VectorField / TensorField objects.

Order of fields determines block ordering.
User is responsible for matching this order
to the SystemMatrix trial ordering.
"""
function SystemVector_(fields::Vector)

    isempty(fields) && error("SystemVector: empty field list.")

    # ----------------------------------------------------------
    # 1) Extract problems in GIVEN order
    # ----------------------------------------------------------
    problems = Problem[]

    for f in fields
        P = f.model
        push!(problems, P)
    end

    # check uniqueness (no duplicate fields)
    length(unique(problems)) == length(problems) ||
        error("SystemVector: duplicate Problems in field list.")

    # ----------------------------------------------------------
    # 2) Compute offsets (trial order)
    # ----------------------------------------------------------
    offsets = Vector{Int}(undef, length(problems))
    offsets[1] = 0

    for i in 2:length(problems)
        offsets[i] = offsets[i-1] + ndofs(problems[i-1])
    end

    total_dofs = offsets[end] + ndofs(problems[end])
    a_global = zeros(total_dofs)

    # ----------------------------------------------------------
    # 3) Fill global vector
    # ----------------------------------------------------------
    for (idx, f) in enumerate(fields)

        P = f.model
        offset = offsets[idx]
        nloc = ndofs(P)

        # single load step assumption
        a_local = vec(f.a[:, 1])

        length(a_local) == nloc ||
            error("SystemVector: local RHS size mismatch for problem $(P.name).")

        a_global[offset+1:offset+nloc] .= a_local
    end

    return SystemVector(a_global, nothing, problems, offsets)
end

SystemVector_

In [4]:
abstract type AbstractOp end

struct IdOp <: AbstractOp end     # u
struct GradOp <: AbstractOp end     # ∇u
struct DivOp <: AbstractOp end     # div u
struct CurlOp <: AbstractOp end    # rot u
struct SymGradOp <: AbstractOp end
struct TensorDivOp <: AbstractOp end
struct AdvOp <: AbstractOp
    b::NTuple{3,Float64}
end

In [5]:
"""
    ndofs(problem::Problem)

Return total number of dofs for a single-field problem.
"""
ndofs(P::Problem) = P.non * P.pdim

"""
    op_outdim(op, P)

Return the number of components produced by applying `op` to a field of `P`.
This determines the row-count of the operator matrix at a Gauss point.
"""
function op_outdim(::IdOp, P::Problem)
    return P.pdim
end

function op_outdim(::GradOp, P::Problem)
    # Scalar: grad -> dim
    if P.pdim == 1
        return P.dim
    end
    # Vector: grad(u) -> dim×pdim (full gradient, column per component)
    return P.dim * P.pdim
end

function op_outdim(::DivOp, P::Problem)
    # Vector: div -> 1 (assume pdim==dim)
    @assert P.pdim == P.dim "DivOp currently assumes vector field with pdim == dim."
    return 1
end

function op_outdim(::CurlOp, P::Problem)
    @assert P.pdim == P.dim "CurlOp requires vector field with pdim == dim."
    return (P.dim == 2) ? 1 : 3
end

function op_outdim(::SymGradOp, P::Problem)
    @assert P.pdim == P.dim "SymGradOp requires vector field with pdim == dim."
    return (P.dim == 2) ? 3 : 6  # engineering strain components
end

function op_outdim(::TensorDivOp, P::Problem)
    @assert P.pdim == P.dim^2 "TensorDivOp requires pdim == dim^2 (full 2nd-order tensor)."
    return P.dim
end

function op_outdim(op::AdvOp, P::Problem)
    @assert P.pdim == 1  # scalar field
    return 1
end

op_outdim (generic function with 7 methods)

In [6]:
"""
    build_B!(B, op, P, k, h, ∂h, numNodes)

Fill the operator matrix `B` at Gauss point `k` for operator `op` and problem `P`.
- `B` has size (op_outdim(op,P), P.pdim*numNodes)
"""
function build_B!(B::AbstractMatrix, ::IdOp, P::Problem, k::Int, h, ∂h, numNodes::Int)
    fill!(B, 0.0)
    pdim = P.pdim
    @inbounds for a in 1:numNodes
        Na = h[(k-1)*numNodes+a]
        @inbounds for c in 1:pdim
            row = c
            col = (a - 1) * pdim + c
            B[row, col] = Na
        end
    end
    return B
end

function build_B!(B::AbstractMatrix, ::GradOp, P::Problem, k::Int, h, ∂h, numNodes::Int)
    fill!(B, 0.0)
    pdim = P.pdim
    dim = P.dim

    if pdim == 1
        # grad(scalar): rows = dim
        @inbounds for a in 1:numNodes
            col = a # scalar: (a-1)*1 + 1
            @inbounds for d in 1:dim
                row = d
                B[row, col] = ∂h[d, (k-1)*numNodes+a]
            end
        end
    else
        # grad(vector): rows = dim*pdim
        # ordering rows: (comp-1)*dim + d   (component-major)
        @inbounds for a in 1:numNodes
            @inbounds for c in 1:pdim
                col = (a - 1) * pdim + c
                @inbounds for d in 1:dim
                    row = (c - 1) * dim + d
                    B[row, col] = ∂h[d, (k-1)*numNodes+a]
                end
            end
        end
    end
    return B
end

function build_B!(B::AbstractMatrix, ::DivOp, P::Problem, k::Int, h, ∂h, numNodes::Int)
    fill!(B, 0.0)
    pdim = P.pdim
    dim = P.dim
    @assert pdim == dim

    # div(u) = Σ_i ∂u_i/∂x_i
    # For basis (node a, component i): contribution is ∂N_a/∂x_i
    @inbounds for a in 1:numNodes
        @inbounds for i in 1:dim
            col = (a - 1) * pdim + i
            B[1, col] += ∂h[i, (k-1)*numNodes+a]
        end
    end
    return B
end

function build_B!(B::AbstractMatrix, ::CurlOp,
    P::Problem, k::Int, h, ∂h, numNodes::Int)
    fill!(B, 0.0)
    dim = P.dim
    pdim = P.pdim
    @assert pdim == dim

    if dim == 2
        # curl(u) = ∂uy/∂x - ∂ux/∂y  (scalar)
        @inbounds for a in 1:numNodes
            colx = (a - 1) * pdim + 1
            coly = (a - 1) * pdim + 2
            B[1, colx] = -∂h[2, (k-1)*numNodes+a]   # -∂N/∂y * ux_a
            B[1, coly] = ∂h[1, (k-1)*numNodes+a]   #  ∂N/∂x * uy_a
        end
    else
        # 3D curl:
        # cx = ∂uz/∂y - ∂uy/∂z
        # cy = ∂ux/∂z - ∂uz/∂x
        # cz = ∂uy/∂x - ∂ux/∂y
        @inbounds for a in 1:numNodes
            colx = (a - 1) * pdim + 1
            coly = (a - 1) * pdim + 2
            colz = (a - 1) * pdim + 3

            dNx = ∂h[1, (k-1)*numNodes+a]
            dNy = ∂h[2, (k-1)*numNodes+a]
            dNz = ∂h[3, (k-1)*numNodes+a]

            # cx row = 1
            B[1, coly] = -dNz
            B[1, colz] = dNy

            # cy row = 2
            B[2, colx] = dNz
            B[2, colz] = -dNx

            # cz row = 3
            B[3, colx] = -dNy
            B[3, coly] = dNx
        end
    end

    return B
end

function build_B!(B::AbstractMatrix, ::SymGradOp,
    P::Problem, k::Int, h, ∂h, numNodes::Int)
    fill!(B, 0.0)
    dim = P.dim
    pdim = P.pdim
    @assert pdim == dim

    if dim == 2
        # rows: [εxx, εyy, γxy]
        @inbounds for a in 1:numNodes
            colx = (a - 1) * pdim + 1
            coly = (a - 1) * pdim + 2
            dNx = ∂h[1, (k-1)*numNodes+a]
            dNy = ∂h[2, (k-1)*numNodes+a]

            B[1, colx] = dNx          # εxx
            B[2, coly] = dNy          # εyy
            B[3, colx] = dNy          # γxy = dux/dy + duy/dx
            B[3, coly] = dNx
        end
    else
        # rows: [εxx, εyy, εzz, γxy, γxz, γyz]
        @inbounds for a in 1:numNodes
            colx = (a - 1) * pdim + 1
            coly = (a - 1) * pdim + 2
            colz = (a - 1) * pdim + 3

            dNx = ∂h[1, (k-1)*numNodes+a]
            dNy = ∂h[2, (k-1)*numNodes+a]
            dNz = ∂h[3, (k-1)*numNodes+a]

            B[1, colx] = dNx          # εxx
            B[2, coly] = dNy          # εyy
            B[3, colz] = dNz          # εzz

            B[4, colx] = dNy          # γxy
            B[4, coly] = dNx

            B[5, colx] = dNz          # γxz
            B[5, colz] = dNx

            B[6, coly] = dNz          # γyz
            B[6, colz] = dNy
        end
    end

    return B
end

function build_B!(B::AbstractMatrix, ::TensorDivOp,
    P::Problem, k::Int, h, ∂h, numNodes::Int)
    fill!(B, 0.0)
    dim = P.dim
    pdim = P.pdim  # dim^2

    # assume tensor component ordering at node:
    # σ_ij mapped to α = (i-1)*dim + j   (i=row, j=col)
    @inbounds for a in 1:numNodes
        for i in 1:dim
            row = i
            for j in 1:dim
                col = (a - 1) * pdim + (i - 1) * dim + j
                B[row, col] = ∂h[j, (k-1)*numNodes+a]  # ∂/∂x_j
            end
        end
    end
    return B
end

function build_B!(B::AbstractMatrix, op::AdvOp,
    P::Problem, k::Int, h, ∂h, numNodes::Int)

    fill!(B, 0.0)
    b = op.b
    dim = P.dim

    @inbounds for a in 1:numNodes
        val = 0.0
        for d in 1:dim
            val += b[d] * ∂h[d, (k-1)*numNodes+a]
        end
        B[1, a] = val
    end

    return B
end

build_B! (generic function with 7 methods)

In [7]:
@inline function _build_elemwise_coeff_dict(coef::ScalarField)
    p = nodesToElements(coef)
    return Dict(zip(p.numElem, p.A))  # elemTag => coeff nodal vector(s)
end

@inline function _coeff_at_gp(pa::Dict{<:Integer,<:AbstractMatrix}, elem::Integer, hcol::AbstractVector)
    return dot(view(pa[elem], :, 1), hcol)
end


_coeff_at_gp (generic function with 1 method)

In [8]:
"""
    assemble_operator(Pu, op_u, Ps, op_s; coefficient=1.0)

Assemble a sparse block matrix for the bilinear form
    ∫ (Op_s v) · (Op_u u) * coefficient dΩ
where trial field is from problem `Pu` and test field is from problem `Ps`.

Currently supports scalar coefficient (Number or ScalarField).
Requires that both problems share the same gmsh model (same mesh).
"""
function assemble_operator(
    Pu::Problem, op_u::AbstractOp,
    Ps::Problem, op_s::AbstractOp;
    coefficient::Union{Number,ScalarField}=1.0,
    weight=nothing)

    @assert Pu.name == Ps.name "Both problems must refer to the same gmsh model/mesh."
    @assert Pu.dim == Ps.dim "Both problems must have the same spatial dimension."
    gmsh.model.setCurrent(Pu.name)

    # output dims must match for the dot-product integrand
    out_u = op_outdim(op_u, Pu)
    out_s = op_outdim(op_s, Ps)
    @assert out_u == out_s "Operator output dims mismatch: $out_u vs $out_s."

    # coefficient prep
    pconst = 1.0
    pa = Dict{Int,Any}()
    if coefficient isa Number
        pconst = Float64(coefficient)
    else
        pa = _build_elemwise_coeff_dict(coefficient)
    end

    # estimate IJV length: crude but OK for notebook prototype
    # worst-case: full dense per element: (pdim_s*numNodes)*(pdim_u*numNodes)
    lengthOfIJV = LowLevelFEM.estimateLengthOfIJV(Pu) * max(1, Ps.pdim) * max(1, Pu.pdim)
    I = Vector{Int}(undef, lengthOfIJV)
    J = Vector{Int}(undef, lengthOfIJV)
    V = Vector{Float64}(undef, lengthOfIJV)
    pos = 1

    dim = Pu.dim

    # loop materials in Pu (same physical names expected)
    for mat in Pu.material
        phName = mat.phName
        dimTags = gmsh.model.getEntitiesForPhysicalName(phName)

        for (edim, etag) in dimTags
            elemTypes, elemTags, elemNodeTags = gmsh.model.mesh.getElements(edim, etag)

            for itype in eachindex(elemTypes)
                et = elemTypes[itype]
                _, _, order, numNodes::Int64, _, _ = gmsh.model.mesh.getElementProperties(et)

                intPoints, intWeights = gmsh.model.mesh.getIntegrationPoints(et, "Gauss" * string(2order + 1))
                numIntPoints = length(intWeights)

                _, fun, _ = gmsh.model.mesh.getBasisFunctions(et, intPoints, "Lagrange")
                h = reshape(fun, :, numIntPoints)

                _, dfun, _ = gmsh.model.mesh.getBasisFunctions(et, intPoints, "GradLagrange")
                ∇h = reshape(dfun, :, numIntPoints)

                # buffers
                nel = length(elemTags[itype])
                nnet = zeros(Int, nel, numNodes)
                invJac = zeros(3, 3numIntPoints)
                ∂h = zeros(dim, numNodes * numIntPoints)

                ndofs_u_loc = Pu.pdim * numNodes
                ndofs_s_loc = Ps.pdim * numNodes

                Bu = zeros(out_u, ndofs_u_loc)
                Bs = zeros(out_s, ndofs_s_loc)
                Ke = zeros(ndofs_s_loc, ndofs_u_loc)

                # connectivity table
                @inbounds for e in 1:nel
                    for a in 1:numNodes
                        nnet[e, a] = elemNodeTags[itype][(e-1)*numNodes+a]
                    end
                end

                tmpBu = similar(Bu)
                # element loop
                @inbounds for e in 1:nel
                    elem = elemTags[itype][e]

                    jac, jacDet, _ = gmsh.model.mesh.getJacobian(elem, intPoints)
                    Jac = reshape(jac, 3, :)

                    @inbounds for k in 1:numIntPoints
                        invJac[1:3, 3k-2:3k] .= inv(Jac[1:3, 3k-2:3k])'
                    end

                    # physical gradients of basis
                    fill!(∂h, 0.0)
                    @inbounds for k in 1:numIntPoints, a in 1:numNodes
                        invJk = invJac[1:dim, 3k-2:3k-(3-dim)]
                        gha = ∇h[a*3-2:a*3-(3-dim), k]
                        ∂h[1:dim, (k-1)*numNodes+a] .= invJk * gha
                    end

                    fill!(Ke, 0.0)

                    # integrate
                    @inbounds for k in 1:numIntPoints
                        cval = (coefficient isa ScalarField) ? _coeff_at_gp(pa, elem, view(h, :, k)) : pconst
                        w = jacDet[k] * intWeights[k] * cval

                        build_B!(Bu, op_u, Pu, k, h, ∂h, numNodes)
                        # --- after build_B!(Bu, ...)
                        # weight: nothing / Vector / Matrix
                        if weight !== nothing
                            if weight isa AbstractVector
                                @inbounds for r in 1:size(Bu, 1)
                                    wr = weight[r]
                                    @inbounds for c in 1:size(Bu, 2)
                                        Bu[r, c] *= wr
                                    end
                                end
                            else
                                # weight isa AbstractMatrix: Bu = weight * Bu
                                # buffer needed: tmp has same size as Bu
                                mul!(tmpBu, weight, Bu)
                                Bu .= tmpBu
                            end
                        end

                        #if weight !== nothing
                        #    @inbounds for r in 1:size(Bu, 1)
                        #        wr = weight[r]
                        #        @inbounds for c in 1:size(Bu, 2)
                        #            Bu[r, c] *= wr
                        #        end
                        #    end
                        #end
                        build_B!(Bs, op_s, Ps, k, h, ∂h, numNodes)

                        # Ke += Bs' * Bu * w
                        mul!(Ke, transpose(Bs), Bu, w, 1.0)
                    end

                    # scatter Ke(s,u) -> global IJV
                    @inbounds for a_loc in 1:ndofs_s_loc
                        node_a = div(a_loc - 1, Ps.pdim) + 1
                        comp_a = mod(a_loc - 1, Ps.pdim) + 1
                        Ia_node = nnet[e, node_a]
                        Ia = (Ia_node - 1) * Ps.pdim + comp_a

                        @inbounds for b_loc in 1:ndofs_u_loc
                            node_b = div(b_loc - 1, Pu.pdim) + 1
                            comp_b = mod(b_loc - 1, Pu.pdim) + 1
                            Jb_node = nnet[e, node_b]
                            Jb = (Jb_node - 1) * Pu.pdim + comp_b

                            if pos > length(I)
                                # grow (not pretty, but notebook-friendly)
                                newlen = Int(ceil(1.5length(I))) + 1000
                                resize!(I, newlen)
                                resize!(J, newlen)
                                resize!(V, newlen)
                            end

                            I[pos] = Ia
                            J[pos] = Jb
                            V[pos] = Ke[a_loc, b_loc]
                            pos += 1
                        end
                    end
                end
            end
        end
    end

    resize!(I, pos - 1)
    resize!(J, pos - 1)
    resize!(V, pos - 1)
    K = sparse(I, J, V, ndofs(Ps), ndofs(Pu))
    dropzeros!(K)
    return SystemMatrix(K, Pu, Ps)
end


assemble_operator

In [9]:
"""
    compliance9_iso(E, nu; penalty=1e8)

Return a 9×9 compliance-like matrix acting on vec(σ) with ordering
(11,12,13,21,22,23,31,32,33). Symmetric part follows isotropic linear elasticity,
antisymmetric part is penalized with `penalty`.
"""
function compliance9_iso(E, nu; penalty=1e8)
    G = E / (2 * (1 + nu))

    # Voigt compliance (engineering shear):
    # [ε11, ε22, ε33, γ12, γ13, γ23] = S6 * [σ11, σ22, σ33, σ12, σ13, σ23]
    S6 = zeros(6, 6)
    S6[1, 1] = 1 / E
    S6[2, 2] = 1 / E
    S6[3, 3] = 1 / E
    S6[1, 2] = -nu / E
    S6[1, 3] = -nu / E
    S6[2, 1] = -nu / E
    S6[2, 3] = -nu / E
    S6[3, 1] = -nu / E
    S6[3, 2] = -nu / E
    S6[4, 4] = 1 / G
    S6[5, 5] = 1 / G
    S6[6, 6] = 1 / G

    # Map 9 -> 6 (take symmetric + engineering shear)
    # v9 = [σ11 σ12 σ13 σ21 σ22 σ23 σ31 σ32 σ33]
    # v6 = [σ11, σ22, σ33, σ12, σ13, σ23] with σ12 := (σ12+σ21)/2 etc.
    P = zeros(6, 9)
    # normals
    P[1, 1] = 1.0       # σ11
    P[2, 5] = 1.0       # σ22
    P[3, 9] = 1.0       # σ33
    # shear sym averages
    P[4, 2] = 0.5
    P[4, 4] = 0.5   # σ12
    P[5, 3] = 0.5
    P[5, 7] = 0.5   # σ13
    P[6, 6] = 0.5
    P[6, 8] = 0.5   # σ23

    # Sym part compliance in 9-space: Ssym = P' * S6 * P
    Ssym = P' * S6 * P

    # Antisym penalty: penalize (σ12-σ21), (σ13-σ31), (σ23-σ32)
    # Add penalty * Q'Q where Q*v9 = [σ12-σ21, σ13-σ31, σ23-σ32]
    Q = zeros(3, 9)
    Q[1, 2] = 1.0
    Q[1, 4] = -1.0
    Q[2, 3] = 1.0
    Q[2, 7] = -1.0
    Q[3, 6] = 1.0
    Q[3, 8] = -1.0
    Santi = penalty * (Q' * Q)

    return Ssym + Santi
end


compliance9_iso

In [10]:
"""
    solveField(K::SystemMatrix, F::SystemVector; support=Vector{BoundaryCondition}())

Solve multi-field linear system K * x = F
with Dirichlet boundary conditions.

Returns one field per Problem in the order stored in K.problems.
"""
function solveField(
    K::SystemMatrix,
    F::SystemVector;
    support::Vector{BoundaryCondition}=BoundaryCondition[]
)

    # ----------------------------------------------------------
    # 1) Consistency checks
    # ----------------------------------------------------------
    K.problems === nothing &&
        error("solveField: SystemMatrix is not a block system.")

    F.problems === nothing &&
        error("solveField: SystemVector is not a block vector.")

    K.problems == F.problems ||
        error("solveField: Problem ordering mismatch between K and F.")

    problems = K.problems
    offsets = K.offsets

    A = K.A
    b = F.a

    ndof = length(b)

    # ----------------------------------------------------------
    # 2) Collect global constrained DOFs
    # ----------------------------------------------------------
    fixed = Int[]

    for bc in support

        P = bc.problem
        idx = findfirst(q -> q === P, problems)

        idx === nothing &&
            error("solveField: BC refers to Problem not in system.")

        offset = offsets[idx]

        local_dofs = constrainedDoFs(P, [bc])

        append!(fixed, offset .+ local_dofs)
    end

    fixed = unique(fixed)
    sort!(fixed)

    # ----------------------------------------------------------
    # 3) Define free DOFs
    # ----------------------------------------------------------
    all_dofs = collect(1:ndof)
    free = setdiff(all_dofs, fixed)

    # ----------------------------------------------------------
    # 4) Reduced system with non-homogeneous BC
    # ----------------------------------------------------------

    x = zeros(ndof)
    xD = zeros(ndof)

    # fill xD
    for bc in support
        P = bc.problem
        idx = findfirst(q -> q === P, problems)
        offset = offsets[idx]

        x_local = applyBoundaryConditions(P, [bc])
        local_dofs = constrainedDoFs(P, [bc])

        xD[offset.+local_dofs] .= x_local.a[local_dofs, 1]
    end

    fixed = unique(vcat([offsets[findfirst(q -> q === bc.problem, problems)] .+
                         constrainedDoFs(bc.problem, [bc])
                         for bc in support]...))

    free = setdiff(1:ndof, fixed)

    A_ff = A[free, free]
    b_f = b[free] - A[free, fixed] * xD[fixed]

    x[free] = A_ff \ b_f
    x[fixed] = xD[fixed]

    # fixed DOFs remain zero (homogeneous)

    # ----------------------------------------------------------
    # 5) Reconstruct fields
    # ----------------------------------------------------------
    results = Vector{Any}(undef, length(problems))

    for (i, P) in enumerate(problems)

        offset = offsets[i]
        nloc = ndofs(P)

        xloc = x[offset+1:offset+nloc]

        if P.pdim == 1
            results[i] = ScalarField([], reshape(xloc, :, 1), [0], [], 1, :scalar, P)

        elseif P.pdim == 2 || P.pdim == 3
            type = P.pdim == 2 ? :v2D : :v3D
            results[i] = VectorField([], reshape(xloc, :, 1), [0], [], 1, type, P)

        elseif P.pdim == 9
            results[i] = TensorField([], reshape(xloc, :, 1), [0], [], 1, :tensor, P)

        else
            error("solveField: unsupported pdim $(P.pdim).")
        end
    end

    return tuple(results...)
end

solveField

In [11]:
gmsh.initialize()

In [12]:
structured_box_mesh()

In [13]:
material = Material("volume")

Pp = Problem([material], type=:ScalarField, field=:p, rhs_field=:q)
Pu = Problem([material], type=:VectorField, field=:u, rhs_field=:f)
Ps = Problem([material], type=:TensorField, field=:s)

Problem("", :TensorField, 3, 9, Material[Material("volume", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1331, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :s, :rhs)

In [14]:
pres = BoundaryCondition("left", problem=Pp, p=2)

BoundaryCondition("left", Problem("", :ScalarField, 3, 1, Material[Material("volume", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1331, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :p, :q), Dict{Symbol, Union{Function, Number, ScalarField}}(:p => 2))

In [15]:
supp = BoundaryCondition("left", problem=Pu, uz=2, uy=1)

BoundaryCondition("left", Problem("", :VectorField, 3, 3, Material[Material("volume", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1331, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f), Dict{Symbol, Union{Function, Number, ScalarField}}(:uz => 2, :uy => 1))

In [16]:
supp3 = BoundaryCondition("left", problem=Ps, sxy=1)

BoundaryCondition("left", Problem("", :TensorField, 3, 9, Material[Material("volume", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1331, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :s, :rhs), Dict{Symbol, Union{Function, Number, ScalarField}}(:sxy => 1))

In [17]:
load = LoadCondition("right", problem=Pu, fx=1)

LoadCondition("right", Problem("", :VectorField, 3, 3, Material[Material("volume", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1331, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f), Dict{Symbol, Union{Function, Number, ScalarField}}(:fx => 1))

In [18]:
f = loadVector(Pu, [load])

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [19]:
g0 = LoadCondition("right", problem=Pp, q=0)
g = loadVector(Pp, [g0])

nodal ScalarField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [20]:
A = assemble_operator(Pu, GradOp(), Pu, GradOp())
B = assemble_operator(Pu, DivOp(), Pp, IdOp())
C = assemble_operator(Pp, IdOp(), Pp, IdOp(), coefficient=0);

In [21]:
K = SystemMatrix([A B'; B C])
K[:, :]

5324×5324 SparseMatrixCSC{Float64, Int64} with 265321 stored entries:
⎡⡿⣯⣿⠟⢿⣤⡟⠀⡇⢈⣥⣼⣷⣯⣤⡧⠀⡇⠐⡇⢘⠄⢲⠀⡢⠀⡆⠰⣷⣾⣿⣿⡿⣽⣿⢾⣿⣖⢶⣿⎤
⎢⣿⠟⠿⣧⡙⠀⢯⠉⡟⠛⠓⠿⠁⠺⠶⠷⠶⢦⣤⣤⣤⣄⣀⣀⣀⣀⡀⠀⠁⢸⣿⣿⢹⠻⠿⢦⣤⣀⡀⠑⎥
⎢⠛⣷⠓⠈⠻⣦⡈⠀⡇⢰⠒⠚⠓⠒⠒⠶⠶⠶⠶⢦⣤⣤⣭⣍⣉⣉⣙⣛⡛⠛⡟⢿⢿⡞⠲⠶⣦⣭⣙⡛⎥
⎢⠛⠉⡏⠓⠂⠈⠻⣦⡁⠈⡇⢰⡉⢹⠀⢷⠀⡇⠀⡇⢰⡀⢰⠀⢻⠉⡏⠉⡏⠙⢿⠚⣏⣿⣿⣸⣷⣾⣹⡏⎥
⎢⡉⢉⣿⠉⢉⣉⡁⠈⠻⣦⡁⢸⠁⢿⠀⡏⠀⡇⢸⡅⢸⠁⢻⠀⡏⠀⡇⢸⡅⢸⣹⣉⢹⣽⣿⢻⣿⣿⢻⣿⎥
⎢⣁⣿⣽⡄⣸⠀⢉⣉⣁⣈⠻⣦⡀⠘⢷⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠁⠀⡁⢈⣿⡏⣁⣷⢻⠀⠀⠀⠈⡉⎥
⎢⡽⣿⣡⡀⢹⠀⣇⣈⣥⣄⣀⠈⠻⣦⡀⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢮⣯⣿⣩⡘⣇⡀⠀⠀⠀⢷⎥
⎢⠤⡿⢼⡇⢸⡄⢤⣄⡤⠤⠙⠷⠀⢨⡻⣮⡻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠠⣿⡧⡤⠿⣽⣧⠀⠀⠀⠀⎥
⎢⠤⠤⠸⣇⢸⡇⠤⠤⠤⠤⠀⠀⠀⠈⠻⣮⡻⣮⡻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠠⢼⣿⠤⠄⢹⣿⡆⠀⠀⠀⎥
⎢⠴⠤⠀⣿⠸⣇⠤⠤⠖⠶⠀⠀⠀⠀⠀⠈⠻⣮⡻⣮⡻⣤⡀⠀⠀⠀⠀⠀⠀⠠⢾⣿⠴⠆⠀⢿⣿⡀⠀⠀⎥
⎢⠒⠔⠀⢿⠀⣿⠐⠲⠖⠒⠀⠀⠀⠀⠀⠀⠀⠈⠛⣮⡻⣮⡻⣦⡀⠀⠀⠀⠀⠐⢾⣿⠶⠂⠀⠘⣿⣧⠀⠀⎥
⎢⠘⠒⠀⢸⡇⢿⠐⠒⠛⠒⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣮⡻⣮⡳⣦⡀⠀⠀⠐⠋⣿⠚⠂⠀⠀⢹⣿⡆⠀⎥
⎢⠈⠊⠀⢸⡇⢸⡟⠒⠋⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠹⣮⡻⣮⡻⣦⡀⠈⠃⣿⠛⠁⠀⠀⠀⢿⣿⡀⎥
⎢⢈⡉⠀⠈⣷⢸⡏⠉⣉⣉⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣮⡻⣮⡻⣮⡁⣿⢉⡁⠀⠀⠀⠘⣿⣧⎥
⎢⣹⣿⣁⣀⣿⠈⣏⠉⣁⣉⡁⢈⡳⣄⠀⡀⠀⡀⠀⡀⢀⠀⢀⠀⡀⠈⡻⣮⡻⣮⣏⣿⢉⡹⡆⠀⠀⠀⢹⣿⎥
⎢⣿⣿⣿⣿⣿⣍⣻⠓⡗⢺⡿⠿⣯⣿⠿⡿⣶⣷⣾⣷⣾⣷⣯⣤⣭⣤⣥⣬⣯⣽⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣟⣯⣷⡒⣻⠷⣯⣽⣗⣶⢥⣼⣃⠺⣤⡏⠀⠇⠰⠇⠸⠃⠺⠀⠟⠀⠇⠰⣇⡰⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣻⣟⠻⣇⢸⡆⣛⣻⣿⣛⠛⠒⠉⠹⠷⣿⣷⣶⣤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢻⢿⠀⢻⡌⣿⣹⣿⣿⣿⠀⠀⠀⠀⠀⠀⠈⠉⠛⠻⠿⣿⣷⣶⣤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⣼⣷⢄⠈⣷⠸⡷⠾⣿⣶⡆⠠⢤⣄⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠻⠿⣿⣷⣶⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [22]:
F = SystemVector([f, g])
F.a;

In [23]:
u, p = solveField(K, F, support=[pres, supp])

(VectorField(Matrix{Float64}[], [3.3915790290446047e13; 1.0; … ; 1.017999557637487; 1.987678279326554;;], [0.0], Int64[], 1, :v3D, Problem("", :VectorField, 3, 3, Material[Material("volume", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1331, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f)), ScalarField(Matrix{Float64}[], [2.0; 2.0; … ; 23.486512489848614; -28.3800569183584;;], [0.0], Int64[], 1, :scalar, Problem("", :ScalarField, 3, 1, Material[Material("volume", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1331, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :p, :q)))

In [24]:
showDoFResults(u, name="u")
showDoFResults(p, name="p")

openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
